# BODAQS data.syn.bike Export - Self-scoped

This notebook selects physical sessions directly from one configured BODAQS library and exports them to data.syn.bike-compatible CSV files.

## 1. Configure Library And Output Path

`OUTPUT_DIR` is deliberately explicit. It may be inside or outside the library root.

In [ ]:
from pathlib import Path
import sys

from IPython.display import display


def find_analysis_dir(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for candidate in (start, *start.parents):
        if (candidate / "bodaqs_analysis").is_dir():
            return candidate
        analysis = candidate / "analysis"
        if (analysis / "bodaqs_analysis").is_dir():
            return analysis
    raise RuntimeError("Could not find the BODAQS analysis package root from the current working directory.")


ANALYSIS_DIR = find_analysis_dir()
if str(ANALYSIS_DIR) not in sys.path:
    sys.path.insert(0, str(ANALYSIS_DIR))

LIBRARIES_ROOT = Path.home() / "OneDrive" / "BODAQS-data"
LIBRARY_ID = "archie"
OUTPUT_DIR = Path.home() / "OneDrive" / "BODAQS-data" / "exports" / "data_syn_bike"

SPLIT_ON_ACTIVITY = True
DROP_INACTIVE = True
INACTIVE_MASK_COLUMN = "active_mask_qc"
TIME_FORMAT = "sample_count"
SAMPLE_COUNT_ORIGIN = "session"
ADC_BIT_COUNT = 12
RAW_SCALING_MODE = "calibrated_full_scale"
FILENAME_TEMPLATE = "{run_id}__{session_id}__{export_id}__data_syn_bike.csv"

from bodaqs_analysis.exporters.data_syn_bike import default_data_syn_bike_export_config
from bodaqs_analysis.library_api import LibraryAdapter

adapter = LibraryAdapter(LIBRARIES_ROOT)
libraries = {item["library_id"]: item for item in adapter.list_libraries()}
if LIBRARY_ID not in libraries:
    available = ", ".join(sorted(libraries)) or "none found"
    raise ValueError(f"Library {LIBRARY_ID!r} was not found. Available libraries: {available}")

LIBRARY_ROOT = Path(libraries[LIBRARY_ID]["root"])
EXPORT_CONFIG = default_data_syn_bike_export_config(
    split_on_activity=SPLIT_ON_ACTIVITY,
    drop_inactive=DROP_INACTIVE,
    inactive_mask_column=INACTIVE_MASK_COLUMN,
    time_format=TIME_FORMAT,
    sample_count_origin=SAMPLE_COUNT_ORIGIN,
    adc_bit_count=ADC_BIT_COUNT,
    raw_scaling_mode=RAW_SCALING_MODE,
    filename_template=FILENAME_TEMPLATE,
)

print(f"Analysis package root: {ANALYSIS_DIR}")
print(f"Library root: {LIBRARY_ROOT}")
print(f"Output path: {OUTPUT_DIR}")


## 2. Select Sessions

In [ ]:
from bodaqs_analysis.widgets.session_selector import make_session_selector

selector = make_session_selector(
    artifacts_dir=LIBRARY_ROOT,
    include_aggregations=False,
    select_first_by_default=True,
    autosave_default=False,
)
display(selector["ui"])


## 3. Export Selected Sessions

In [ ]:
from bodaqs_analysis.artifacts import ArtifactStore, load_session_artifacts
from bodaqs_analysis.exporters.data_syn_bike import (
    export_data_syn_bike_resolved,
    write_data_syn_bike_exports,
)

store = ArtifactStore(LIBRARY_ROOT)
selected = selector["get_key_to_ref"]()
if not selected:
    raise ValueError("No sessions selected. Select one or more sessions first.")

written = []
for session_key, (run_id, session_id) in selected.items():
    artifacts = load_session_artifacts(store, run_id=run_id, session_id=session_id)
    session = {
        "run_id": run_id,
        "session_id": session_id,
        "meta": artifacts.get("meta", {}),
        "df": artifacts["df"],
    }
    export_result = export_data_syn_bike_resolved(session, export_config=EXPORT_CONFIG)
    write_result = write_data_syn_bike_exports(export_result, OUTPUT_DIR)
    written.extend(write_result["written"])

    summary = export_result["summary"]
    print(f"Session {run_id} / {session_id}: {summary['n_exports']} export(s), {summary['exported_rows']} row(s)")
    print(
        "  raw columns: "
        f"front={summary['front_raw_col']} inverted={summary['front_raw_inverted']} "
        f"[{summary['front_raw_inversion_reason']}] ({summary['front_raw_reason']}), "
        f"rear={summary['rear_raw_col']} inverted={summary['rear_raw_inverted']} "
        f"[{summary['rear_raw_inversion_reason']}] ({summary['rear_raw_reason']})"
    )
    print(f"  inactive rows dropped: {summary['inactive_rows_dropped']} of {summary['input_rows']}")
    print(
        f"  time format: {summary['time_format']} ({summary['sample_count_origin']})\n"
        "  gps columns: "
        f"lat={summary['lat_col']}, lon={summary['lon_col']}, speed={summary['speed_col']}"
    )

for item in written:
    print(f"Wrote {item['path']} ({item['rows']} rows)")

print(f"\nExport complete: {len(written)} file(s) in {OUTPUT_DIR.resolve()}")
